In [1]:
from stable_baselines3 import PPO
import numpy as np
import pandas as pd
from typing import Any
from OPSDesignerEnv import OPSDesignerEnv

C:\Users\noton\miniconda3\envs\OPSdesign2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead


C:\Users\noton\miniconda3\envs\OPSdesign2\Lib\site-packages\deepchem\feat\smiles_tokenizer.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
fragment_pool = pd.read_csv("source/SMILES_list_100.csv")["SMILES"].tolist()
targetOPS = [-5.9155, -2.7331, 2.5561, 0.0639, 2.4273, 0.1288]

eval_env = OPSDesignerEnv(smiles=fragment_pool,
                          target_ops=targetOPS,
                          min_k=1,
                          max_k=3,
                          max_steps=4,
                          eps=1e-8,
                          r_cap=100,
                         )

def generate_SMILES(model, env, n_episodes=200, base_seed=42, deterministic=False):
    rows = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=base_seed + ep)
        terminated = truncated = False
        total_reward = 0.0
        last_info = {}

        while not (terminated or truncated):
            action, _ = model.predict(obs, deterministic=deterministic)
            obs, r, terminated, truncated, info = env.step(int(action))
            total_reward += float(r)
            last_info = info

        best = last_info.get("best_smiles")
        selected = last_info.get("selected_smiles")
        smiles = best if best is not None else selected

        if isinstance(smiles, list):
            smiles = tuple(smiles)
        if isinstance(selected, list):
            selected = tuple(selected)

        rows.append({"episode": ep, "smiles": smiles, "selected_fragments": selected, "reward": total_reward})
        #print(f"episode {ep} done.")
        
    return pd.DataFrame(rows)

In [3]:
model = PPO.load("RLmodels/ppo_rm_woTL.zip", device="cpu")
df = generate_SMILES(model, eval_env, n_episodes=50)
df.to_csv("generated_OPSs/rl_candidates_woTL.csv", index=False)
df

,episode,smiles,selected_fragments,reward
0,0,O=C1c2ccccc2-c2c1cccc2-n1c2ccccc2c2cc(N3C(=O)c...,"(O=C1c2ccccc2-c2ccccc21, c1ccc2c(c1)Nc1ccccc12...",14.137907
1,1,O=C1c2cccc3c(-n4c5ccccc5c5ccccc54)ccc(c23)C(=O...,"(c1ccc2c(c1)Nc1ccccc12, O=C1NC(=O)c2cccc3cccc1...",15.900213
2,2,O=C1NC(=O)c2ccc(-n3c4ccccc4c4ccc(-n5c6ccccc6c6...,"(c1ccc2c(c1)Nc1ccccc12, c1ccc2c(c1)Nc1ccccc12,...",8.659897
3,3,N#Cc1ccc(-n2c3ccccc3c3cc(-n4c5ccccc5c5ccccc54)...,"(N#Cc1cccs1, c1ccc2c(c1)Nc1ccccc12, c1ccc2c(c1...",5.368524
4,4,c1ccc2c(c1)[nH]c1cc(-n3c4ccccc4c4cc(-n5c6ccccc...,"(c1ccc2c(c1)Nc1ccccc12, c1ccc2c(c1)Nc1ccccc12,...",1.347957
5,5,Cc1cc(C#N)cc(C)c1N1C(=O)c2ccc(-c3cn(C)c4ccccc3...,"(Cn1ccc2ccccc21, O=C1NC(=O)c2ccccc21, Cc1cc(C)...",12.071839
6,6,O=C1c2cccc3c(-n4c5ccccc5c5ccccc54)ccc(c23)C(=O...,"(O=C1NC(=O)c2cccc3cccc1c23, c1ccc2c(c1)Nc1cccc...",15.896087
7,7,O=C1NC(=O)c2ccc(N3C(=O)c4cccc5c(-n6c7ccccc7c7c...,"(c1ccc2c(c1)Nc1ccccc12, O=C1NC(=O)c2cccc3cccc1...",18.804661
8,8,O=C1c2ccc(N3c4ccccc4CCc4ccccc43)cc2C(=O)N1c1cc...,"(c1ccc2c(c1)CCc1ccccc1N2, O=C1NC(=O)c2ccccc21,...",7.401831
9,9,O=C1NC(=O)c2cc(N3C(=O)c4cccc(-n5c6ccccc6c6cccc...,"(O=C1NC(=O)c2ccccc21, c1ccc2c(c1)Nc1ccccc12, O...",14.939200
